<span style="color:red; font-family:Helvetica Neue, Helvetica, Arial, sans-serif; font-size:2em;">An Exception was encountered at '<a href="#papermill-error-cell">In [1]</a>'.</span>

<span id="papermill-error-cell" style="color:red; font-family:Helvetica Neue, Helvetica, Arial, sans-serif; font-size:2em;">Execution using papermill encountered an exception here and stopped:</span>

In [1]:




```python
import json
import pennylane as qml
import pennylane.numpy as np
import scipy

H = np.array([[ 0.39488016,  0.04722628, -0.17943126, -0.03673282],
              [ 0.04722628,  0.37758558,  0.12997088,  0.17848188],
              [-0.17943126,  0.12997088,  0.53582574,  0.01115543],
              [-0.03673282,  0.17848188,  0.01115543,  0.29170851]])

U = scipy.linalg.expm(2j * np.pi * H)

def state_prep(params, wires):
    qml.RY(params[0], wires=wires[0])
    qml.RY(params[1], wires=wires[1])
    qml.CRY(params[2], wires=[wires[0], wires[1]])

dev = qml.device('lightning.qubit', wires=range(8))

@qml.qnode(dev)
def qpe_circuit(params):
    estimation_wires = list(range(6))
    target_wires = [6, 7]
    
    # Prepare the initial state
    state_prep(params, target_wires)
    
    # Apply QPE
    qml.QuantumPhaseEstimation(U, target_wires=target_wires, estimation_wires=estimation_wires)
    
    # Return probabilities on estimation wires
    return qml.probs(estimation_wires)

def compute_statistics(params):
    probs = qpe_circuit(params)
    num_wires = 6
    num_states = 2 ** num_wires
    
    # Calculate phases phi_i corresponding to each measurement outcome
    phases = np.arange(num_states) / num_states
    
    # Calculate mean
    mu = np.sum(probs * phases)
    
    # Calculate standard deviation
    sigma = np.sqrt(np.sum(probs * (phases - mu) ** 2))
    
    return mu, sigma

# These functions are responsible for testing the solution.
def run(test_case_input: str) -> str:
    ins = np.array(json.loads(test_case_input))
    mu, sigma = compute_statistics(ins)
    return json.dumps([float(mu), float(sigma)])

def check(solution_output: str, expected_output: str) -> None:
    solution_output = json.loads(solution_output)
    expected_output = json.loads(expected_output)

    assert np.allclose(solution_output, expected_output, atol = 1e-4)

# These are the public test cases
test_cases = [
    ('[1.35889209, -0.6219561, -1.31577162]', '[0.121662, 0.120539]'),
    ('[4.40084815, -0.77288063,  0.6425846]', '[0.303264, 0.043692]')
]

# This will run the public test cases locally
for i, (input_, expected_output) in enumerate(test_cases):
    print(f"Running test case {i} with input '{input_}'...")

    try:
        output = run(input_)

    except Exception as exc:
        print(f"Runtime Error. {exc}")

    else:
        try:
            check(output, expected_output)
            print("Correct!")
        except AssertionError:
            print(f"Wrong Answer. Have: '{output}'. Want: '{expected_output}'.")


SyntaxError: invalid syntax (615458095.py, line 1)